In [1]:
import pandas as pd
import joblib

final_model = joblib.load("models/final_model.pkl")
master = pd.read_csv("data/processed/master_features.csv")

feature_cols = ["age", "max_gap_days", "total_time_min", "quiz_completion_pct", "avg_quiz_score"]

In [2]:
def assign_risk_tier(probability):
    if probability >= 0.70:
        return "Low"
    elif probability >= 0.40:
        return "Medium"
    else:
        return "High"

In [3]:
completer_avg = master[master["dropout_type"] == "completer"][feature_cols].mean()

def get_top_reasons(student_row, n=3):
    reasons = []
    diffs = {
        "max_gap_days": (student_row["max_gap_days"] - completer_avg["max_gap_days"], "day inactivity gap"),
        "total_time_min": (completer_avg["total_time_min"] - student_row["total_time_min"], "min less time on course"),
        "quiz_completion_pct": (completer_avg["quiz_completion_pct"] - student_row["quiz_completion_pct"], "% lower quiz completion"),
        "avg_quiz_score": (completer_avg["avg_quiz_score"] - student_row["avg_quiz_score"], "points lower avg quiz score"),
    }
    sorted_reasons = sorted(diffs.items(), key=lambda x: x[1][0], reverse=True)
    for feature, (value, label) in sorted_reasons[:n]:
        if value > 0:
            reasons.append(f"{round(value,1)} {label}")
    return reasons

In [8]:
def predict_completion_probability(student_row):
    X = student_row[feature_cols].to_frame().T   # keep it as a DataFrame, not .values
    prob = final_model.predict_proba(X)[0][1]
    tier = assign_risk_tier(prob)
    reasons = get_top_reasons(student_row) if tier != "Low" else []
    return {
        "student_id": student_row["student_id"],
        "completion_probability": round(prob * 100, 1),
        "risk_tier": tier,
        "top_reasons": reasons,
    }

In [9]:
for i in [0, 50, 900]:  # pick a few random rows
    result = predict_completion_probability(master.iloc[i])
    print(result)

{'student_id': 'S00001', 'completion_probability': np.float64(100.0), 'risk_tier': 'Low', 'top_reasons': []}
{'student_id': 'S00051', 'completion_probability': np.float64(0.0), 'risk_tier': 'High', 'top_reasons': ['922.7 min less time on course', '80.0 % lower quiz completion', '23.2 points lower avg quiz score']}
{'student_id': 'S00901', 'completion_probability': np.float64(100.0), 'risk_tier': 'Low', 'top_reasons': []}
